(defexpr)=
# Deferred Expressions and Contexts

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

Consider the simple FODO cell:

In [2]:
with sb.elements():
    qf = sb.Quadrupole(Kn1=0.36, L=0.5)
    d = sb.Drift(L=1.2)
    qd = sb.Quadrupole(Kn1=-0.36, L=0.5)

fodo = sb.Beamline([qf, d, qd, d], E_ref=18e9,
                   species_ref=sb.Species("electron"))
fodo

Beamline:
 species_ref = electron
 E_ref = 1.8e10

  Index   Name   Kind         L [m]   s [m]   s_downstream [m]
  1       qf     Quadrupole   0.5     0.0f0   0.5
  2       d      Drift        1.2     0.5     1.7
  3       qd     Quadrupole   0.5     1.7     2.2
  4       d      Drift        1.2     2.2     3.4000000000000004


We set `qf.Kn1 = 0.36`, and `qd.Kn1 = -0.36`. But what if we want to ensure that
`qd.Kn1 == -qf.Kn1` always? We can bake-in such an interdependence, common in accelerator
parameters, using a "deferred expression" - an expression where evaluation is postponed
until its result is actually needed, rather than immediately when it is defined. To do this, let's first define a variable `kq = 0.36` in an object called a `Context`. For interactive use, there is a global context `DEFAULT_CONTEXT` available:

In [3]:
print("BEFORE:")
print(sb.DEFAULT_CONTEXT)

sb.DEFAULT_CONTEXT.kq = 0.36  # Set variable kq

print("AFTER:")
print(sb.DEFAULT_CONTEXT)

BEFORE:
Context{Any} with 0 variables:

AFTER:
Context{Any} with 1 variable:
 kq = 0.36


:::{only} julia
Then, let's write a [lambda/anonymous function](https://docs.julialang.org/en/v1/manual/functions/#man-anonymous-functions) that receives the context as an argument (below, `c`) and returns either `+kq` or `-kq`:
:::
:::{only} python
Then, let's write a [lambda
function](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions) that receives the
context as an argument (below, `c`) and returns either `+kq` or `-kq`:
:::

In [4]:
# Define lambda functions
kqf = lambda c: c.kq
kqd = lambda c: -c.kq

# Call with the context:
print("kqf(DEFAULT_CONTEXT) =", kqf(sb.DEFAULT_CONTEXT))
print("kqd(DEFAULT_CONTEXT) =", kqd(sb.DEFAULT_CONTEXT))

kqf(DEFAULT_CONTEXT) = 0.36
kqd(DEFAULT_CONTEXT) = -0.36


Now, if we can just set the quadrupole strengths `Kn1` of `qf` and `qd` to always be the outputs of these functions, then we have achieved our goal! SciBmad's `DefExpr` type allows you to do this, by wrapping such lambda functions:

In [5]:
qf.Kn1 = sb.DefExpr(kqf)
qd.Kn1 = sb.DefExpr(kqd)

print("qf.Kn1 =", qf.Kn1)
print("qd.Kn1 =", qd.Kn1)

qf.Kn1 = 0.36
qd.Kn1 = -0.36


:::{only} python
`sb.DefExpr` calls the Python function once, with a context whose variables are symbols, and
keeps the Julia expression that the function builds from them: when the expression is
evaluated later, no Python is called, and it is as fast as one written in Julia. This works
for arithmetic and NumPy functions of the variables (`np.sin(c.kq)`), and for entries of
vector variables (`c.dk[i]`, counted from 0 as in Python). A function that does more, such as
`math.sin(c.kq)` or an `if` on a variable, is called from Julia each time instead, which is
much slower (a `PerformanceWarning` says so). Python values that the function uses other than
the context variables are read when `sb.DefExpr` is called.
:::

When we change the value of `kq`, it is reflected now in the quadrupoles

In [6]:
sb.DEFAULT_CONTEXT.kq = 0.37
print("qf.Kn1 =", qf.Kn1)
print("qd.Kn1 =", qd.Kn1)

qf.Kn1 = 0.37
qd.Kn1 = -0.37


To try a value for a while, {jl}`push!`{py}`sb.push_` it onto the context and
{jl}`pop!`{py}`sb.pop_` it off again: the previous
value is used again (and a variable that {jl}`push!`{py}`sb.push_` added is removed). For example:

In [7]:
sb.DEFAULT_CONTEXT.vrf = 1e6
cav = sb.RFCavity(L=0.1, voltage=sb.DefExpr(lambda c: c.vrf), rf_frequency=5e8)
print("Before push_ :", cav.voltage)
sb.push_(sb.DEFAULT_CONTEXT, vrf=2e6)
print("After push_  :", cav.voltage)
sb.pop_(sb.DEFAULT_CONTEXT)
print("After pop_   :", cav.voltage)

Before push_ : 1000000.0
After push_  : 2000000.0
After pop_   : 1000000.0


 1000000.0
After push_  : 2000000.0
After pop_   : 1000000.0


Alternatively, `withvalues` does the same around a block of code{pytext}` (a with-block)`, and
restores the values even if the code throws an error:

In [8]:
with sb.withvalues(sb.DEFAULT_CONTEXT, vrf=2.0e6):
    print("In withvalues block     :", cav.voltage)
print("Out of withvalues block :", cav.voltage)

In withvalues block     : 2000000.0
Out of withvalues block : 1000000.0


Thus far, we have only been using the global `DEFAULT_CONTEXT`. However, all `Beamline`s have their
own `context` property to store beamline-specific parameters, which the deferred expressions of the
elements in the beamline are evaluated with. If a variable is not found in the beamline context, it
will fall back to `DEFAULT_CONTEXT` if the variable is defined there. All of the above machinery
({jl}`push!`{py}`push_`, {jl}`pop!`{py}`pop_`, `withvalues`) also works with individual beamline
contexts.

In [9]:
sb.DEFAULT_CONTEXT.kq = 0.1

qf = sb.Quadrupole(Kn1=sb.DefExpr(lambda c: c.kq), L=0.5)
bl = sb.Beamline([qf])

child_qf = bl[qf]
print("Before setting Beamline context ---------------")
print("Parent `qf` (not in a beamline):", qf.Kn1)
print("Child `qf` in the beamline:     ", child_qf.Kn1)

# Set kq in the beamline context
sb.push_(bl.context, kq=0.2)

print("After setting Beamline context ----------------")
print("Parent `qf` (not in a beamline):", qf.Kn1)
print("Child `qf` in the beamline:     ", child_qf.Kn1)

Before setting Beamline context ---------------
Parent `qf` (not in a beamline): 0.1
Child `qf` in the beamline:      0.1
After setting Beamline context ----------------
Parent `qf` (not in a beamline): 0.1
Child `qf` in the beamline:      0.2


```{docstring} Context
```

```{docstring} withvalues
```

```{docstring} DefExpr
```

## Getting the Deferred Expression Itself

Getting a parameter of an element always gives its current value: any deferred expression is
evaluated first, as above. Sometimes you want the deferred expression itself instead, for
example to build a new one from it. `rawget` gets a parameter exactly as it was set, and
`deval` evaluates it in a context:

In [10]:
d = sb.rawget(qf, "Kn1")
print(isinstance(d, sb.DefExpr))
print(sb.deval(d, bl.context))  # The same as bl[qf].Kn1

True
0.2


For example, to make `qf` 10% stronger while it keeps following `kq` in the context:

In [11]:
qf.Kn1 = 1.1*sb.rawget(qf, "Kn1")
print(bl[qf].Kn1)
bl.context.kq = 0.4
print(bl[qf].Kn1)

0.22000000000000003
0.44000000000000006


Only the parameters stored as independent variables (set explicitly by the user) can be gotten this way. Others are computed from them: here `Kn1` was set, so e.g. the integrated strength `Kn1L` and the field gradient `Bn1` are computed from `Kn1` (and the length or the reference energy). `rawget` gives an error for these, saying which parameter to use instead:

In [12]:
sb.rawget(qf, "Kn1L")

JuliaRuntimeError: Unable to rawget Kn1L: Kn1L is a dependent variable, computed from the independent variable Kn1. Use rawget with :Kn1 instead

```{docstring} rawget
```

```{docstring} deval
```